# Black Summer — PR from ERA5 daily mx2t (factual) + CMIP6 hist-nat (counterfactual)

Computes Probability Ratio (PR) for extreme fire-season heat in SE Australia using the proper WWA methodology:

- **P1 (factual)**: ERA5 `maximum_2m_temperature` (mx2t) **daily data** → monthly mean of daily max → fire-season max anomaly, 1961–2020
- **P0 (counterfactual)**: CMIP6 `hist-nat` (natural forcing only) pooled anomalies, r1i1p1f1 only

Both P1 and P0 use the same metric: **monthly mean of daily maximum 2m temperature**, matching the CMIP6 `tasmax` definition exactly.

This corrects the two problems from earlier attempts:
1. **ERA5 replaces CMIP6 historical for P1** — CMIP6 historical underestimates Australian warming; ERA5 is the actual observed record
2. **Daily mx2t replaces monthly mean t2m** — monthly mean temperature smooths out extremes and understates the PR signal
3. **r1i1p1f1 only for hist-nat** — eliminates the IPSL-CM6A-LR 10-member imbalance artefact

**Variable**: ERA5 `maximum_2m_temperature` (mx2t) at 06:00 UTC = 24-hour max ending 06 UTC (covers Australian afternoon peak)  
**Metric**: Monthly mean of daily max → Oct–Mar fire-season maximum anomaly vs 1961–1990  
**Area**: SE Australia — 28°S–44°S, 138°E–154°E  
**Validation target**: WWA PR ≥ 10 (heat component of Black Summer)  

## Method

1. Download ERA5 daily mx2t for SE Australia via CDS API (cache to disk, ~200–400 MB)
2. Area-weight spatial mean → daily time series → monthly mean of daily max
3. Compute Oct–Mar seasonal maximum → fire-season anomaly series → **P1 pool**
4. Stream CMIP6 hist-nat (r1i1p1f1) from local pangeo catalog → fire-season anomalies → **P0 pool**
5. Fit Gaussian distributions, compute PR at multiple thresholds
6. Bootstrap 5–95th percentile uncertainty
7. Update Black Summer liability parquet with ERA5-anchored PR columns

## Data Attribution

ERA5 data used in this notebook is provided under the
[Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/) licence.

**Citation**: Hersbach, H., Bell, B., Berrisford, P., et al. (2023): ERA5 monthly averaged data on single levels
from 1940 to present. Copernicus Climate Change Service (C3S) Climate Data Store (CDS).
DOI: [10.24381/cds.f17050d7](https://doi.org/10.24381/cds.f17050d7)

Contains modified Copernicus Climate Change Service information [2026]. Neither the European Commission
nor ECMWF is responsible for any use that may be made of the Copernicus information or data it contains.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger('gcsfs').setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import xarray as xr
import intake
import cdsapi
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import norm
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RAW   = Path('../../data/raw')
PROC  = Path('../../data/processed')
FIGS  = Path('../../outputs/figures')

# SE Australia bounding box
LAT_S, LAT_N = -44, -28
LON_W, LON_E = 138, 154

# Climatology baseline
CLIM_START, CLIM_END = 1961, 1990

# Fire season months (Oct–Mar, spanning two calendar years)
FIRE_MONTHS = [10, 11, 12, 1, 2, 3]

# Analysis period
YEARS = list(range(1961, 2021))  # 1961–2020 inclusive

# Daily ERA5 mx2t — 06:00 UTC timestep = 24-hour max ending 06 UTC
# Covers ~20:00–20:00 AEST, capturing the Australian afternoon maximum
ERA5_PATH = RAW / 'era5' / 'era5_mx2t_daily_se_australia_1961_2020.nc'

# Per-store timeout for CMIP6 streaming
STORE_TIMEOUT = 300

print('Setup complete.')
print(f'ERA5 cache path: {ERA5_PATH}')
print(f'ERA5 already downloaded: {ERA5_PATH.exists()}')

## 1. Download ERA5 daily mx2t via CDS API

Downloads `maximum_2m_temperature` (mx2t) from `reanalysis-era5-single-levels` at the
**06:00 UTC** timestep — this is the 24-hour maximum from 06 UTC the previous day to 06 UTC today,
which encompasses the hottest part of the Australian day (AEST ≈ 16:00–16:00).

Requesting only fire-season months (Oct–Mar) reduces the download to ~200–400 MB for the SE Australia
bounding box at 0.25° resolution over 60 years.

Processing pipeline: daily mx2t → area-weighted spatial mean → monthly mean of daily max →
fire-season (Oct–Mar) seasonal max → anomaly vs 1961–1990.

In [ ]:
if ERA5_PATH.exists():
    print(f'ERA5 file already exists ({ERA5_PATH.stat().st_size / 1e6:.1f} MB) — skipping download.')
else:
    print('Initiating CDS download (daily mx2t, fire-season months, 1961–2020)...')
    print('Expected size: ~200–400 MB compressed. May take 5–20 minutes.')
    ERA5_PATH.parent.mkdir(parents=True, exist_ok=True)

    c = cdsapi.Client()
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': 'maximum_2m_temperature',
            'year':  [str(y) for y in YEARS],
            'month': [f'{m:02d}' for m in FIRE_MONTHS],
            'day':   [f'{d:02d}' for d in range(1, 32)],  # CDS skips invalid dates
            'time':  '06:00',  # 24-h max ending 06 UTC ≈ covers AU afternoon peak
            'data_format': 'netcdf',
            'area':  [LAT_N, LON_W, LAT_S, LON_E],  # [N, W, S, E]
        },
        str(ERA5_PATH)
    )
    print(f'Download complete — {ERA5_PATH.stat().st_size / 1e6:.1f} MB')

## 2. Process ERA5 daily mx2t → P1 (factual) distribution

1. Load daily mx2t, area-weighted spatial mean over SE Australia
2. Resample daily → **monthly mean of daily max** (matches CMIP6 `tasmax` definition)
3. Compute Oct–Mar fire-season maximum (grouped by October year)
4. Anomaly relative to 1961–1990 climatological mean

The 2019 fire season (Oct 2019–Mar 2020) is the event of interest.

In [ ]:
ds_era5 = xr.open_dataset(ERA5_PATH)
print('ERA5 dataset:')
print(ds_era5)

# Identify the mx2t variable (name varies by CDS version)
var_name = None
for candidate in ['mx2t', 'MX2T', 'VAR_228236', 'maximum_2m_temperature', 'MX2T6']:
    if candidate in ds_era5:
        var_name = candidate
        break
if var_name is None:
    var_name = [v for v in ds_era5.data_vars][0]
print(f'\nUsing variable: {var_name}')

da = ds_era5[var_name]  # shape: (time, lat, lon) — daily
print(f'Shape: {da.shape}  ({da.sizes})')

# Detect coordinate names (CDS uses 'latitude'/'longitude' or 'lat'/'lon')
lat_dim = 'latitude' if 'latitude' in da.dims else 'lat'
lon_dim = 'longitude' if 'longitude' in da.dims else 'lon'

# Area-weighted spatial mean
weights = np.cos(np.deg2rad(da[lat_dim])).broadcast_like(da)
ts_daily = da.weighted(weights).mean(dim=[lat_dim, lon_dim]).squeeze()

# Convert K → °C if needed
if float(ts_daily.mean()) > 100:
    ts_daily = ts_daily - 273.15
    print('Converted K → °C')

ts_daily = ts_daily.to_series()
ts_daily.index = pd.to_datetime(ts_daily.index)
ts_daily = ts_daily.dropna()

print(f'\nDaily ERA5 mx2t series: {len(ts_daily)} days ({ts_daily.index.min().date()} – {ts_daily.index.max().date()})')
print(f'Mean: {ts_daily.mean():.1f} °C   Min: {ts_daily.min():.1f}   Max: {ts_daily.max():.1f}')

# Resample to monthly mean of daily max (matches CMIP6 tasmax definition)
ts_monthly = ts_daily.resample('ME').mean()
print(f'Monthly means: {len(ts_monthly)} months')

In [ ]:
# Fire-season (Oct–Mar) maximum, indexed by October year
# ts_monthly is now monthly mean of daily max (matches CMIP6 tasmax)
# Keep only fire-season months before computing seasonal max
ts_fs = ts_monthly[ts_monthly.index.month.isin(FIRE_MONTHS)].copy()

# Shift by 9 months so Oct → Jan → resample gives one value per fire season
ts_shifted = ts_fs.copy()
ts_shifted.index = ts_shifted.index - pd.DateOffset(months=9)
fire_season_era5 = ts_shifted.resample('YE').max()
fire_season_era5.index = fire_season_era5.index.year
fire_season_era5 = fire_season_era5.dropna()

print(f'Fire-season series: {len(fire_season_era5)} seasons ({fire_season_era5.index.min()}–{fire_season_era5.index.max()})')
print(f'  Hottest season:  {fire_season_era5.idxmax()} — {fire_season_era5.max():.2f} °C')
print(f'  2019 season:     {fire_season_era5.get(2019, float("nan")):.2f} °C  ← Black Summer')

# Anomaly vs 1961–1990 climatology
clim_mean = fire_season_era5.loc[CLIM_START:CLIM_END].mean()
era5_anom = fire_season_era5 - clim_mean

print(f'\n1961–1990 climatological mean: {clim_mean:.2f} °C')
print(f'2019 anomaly: {era5_anom.get(2019, float("nan")):.2f} °C  ← event threshold')
print(f'2019 is the hottest season in ERA5 record: {fire_season_era5.idxmax() == 2019}')

# P1 pool
p1_pool = era5_anom.values
print(f'\nP1 pool size: {len(p1_pool)} annual anomalies')

## 3. CMIP6 hist-nat → P0 (counterfactual) distribution

Uses the local pangeo catalog (patched to avoid GCS download).

**Member selection**: `r1i1p1f1` only — one member per model — to avoid the IPSL-CM6A-LR
10-member imbalance that caused the null result in notebook 03. With 4 models × ~60 years each,
we still get ~240 pooled anomalies for P0.

We do **not** need CMIP6 historical here — ERA5 is our P1 source.

In [ ]:
CATALOG_LOCAL = Path('../../data/processed/pangeo-cmip6.json').resolve()
MODELS = ['BCC-CSM2-MR', 'GFDL-ESM4', 'IPSL-CM6A-LR', 'MRI-ESM2-0']

print(f'Loading catalog: {CATALOG_LOCAL}')
col = intake.open_esm_datastore(str(CATALOG_LOCAL))
print('Catalog loaded.')

# hist-nat, r1i1p1f1 only
cat_nat = col.search(
    variable_id='tasmax',
    experiment_id='hist-nat',
    table_id='Amon',
    source_id=MODELS,
    member_id='r1i1p1f1',
)

print(f'\nhist-nat r1i1p1f1 entries: {len(cat_nat.df)}')
print(cat_nat.df[['source_id', 'member_id', 'zstore']].to_string(index=False))

In [ ]:
def au_area_mean_tasmax(zstore):
    """Stream SE Australia area-weighted monthly tasmax from a zarr store."""
    # anon=True: pangeo CMIP6 bucket is public; skip ADC credential check
    ds = xr.open_zarr(zstore, consolidated=True, storage_options={'anon': True})
    da = ds['tasmax']

    extra = [d for d in da.dims if d not in ('time', 'lat', 'lon')]
    if extra:
        da = da.isel({d: 0 for d in extra})

    # Normalise longitudes to 0–360
    if da.lon.values.min() < 0:
        da = da.assign_coords(lon=(da.lon % 360)).sortby('lon')

    da = da.sel(lat=slice(LAT_S, LAT_N), lon=slice(LON_W, LON_E))
    weights = np.cos(np.deg2rad(da.lat)).broadcast_like(da)
    return da.weighted(weights).mean(dim=['lat', 'lon']).squeeze().load()


def fire_season_anom(ts_monthly, clim_start=CLIM_START, clim_end=CLIM_END):
    """Monthly tasmax → fire-season max anomaly vs 1961–1990. Returns annual series."""
    s = ts_monthly.to_series()
    # Some CMIP6 models use non-standard calendars (e.g. NoLeap); convert via str
    # to avoid pd.to_datetime failing on cftime objects
    s.index = pd.DatetimeIndex([pd.Timestamp(str(t)) for t in s.index])
    # Filter to fire-season months only
    s = s[s.index.month.isin(FIRE_MONTHS)]
    # Shift by 9 months so Oct→Jan; resample by year to get fire-season max
    s_shifted = s.copy()
    s_shifted.index = s_shifted.index - pd.DateOffset(months=9)
    seasonal = s_shifted.resample('YE').max().dropna()
    seasonal.index = seasonal.index.year
    # Anomaly vs climatology
    clim = seasonal.loc[clim_start:clim_end].mean()
    return seasonal - clim


def _process_store(zstore):
    ts = au_area_mean_tasmax(zstore)
    return fire_season_anom(ts)


nat_anomalies = []
print(f'Streaming {len(cat_nat.df)} hist-nat stores (r1i1p1f1 only)...\n')

for _, row in cat_nat.df.iterrows():
    tag = f"{row['source_id']} {row['member_id']}"
    print(f'  {tag}...', end=' ', flush=True)
    try:
        with ThreadPoolExecutor(max_workers=1) as ex:
            future = ex.submit(_process_store, row['zstore'])
            anom = future.result(timeout=STORE_TIMEOUT)
        nat_anomalies.append(anom.values)
        print(f'n={len(anom)} years, mean={anom.mean():.2f} K, std={anom.std():.2f}')
    except FuturesTimeoutError:
        print(f'TIMEOUT (>{STORE_TIMEOUT}s) — skipping')
    except Exception as e:
        print(f'FAILED: {e}')

p0_pool = np.concatenate(nat_anomalies) if nat_anomalies else np.array([])
print(f'\nP0 pool: {len(p0_pool)} annual anomalies from {len(nat_anomalies)} model-members')

## 4. Distribution fitting and PR

Fit Gaussian to each pool. Compute PR at multiple thresholds anchored to the ERA5 P1 distribution.
The key threshold is the observed 2019 Black Summer anomaly from ERA5.

In [ ]:
# Fit Gaussians
mu_p1, sigma_p1 = norm.fit(p1_pool)
mu_p0, sigma_p0 = norm.fit(p0_pool)

print('Fitted Gaussian distributions (fire-season tasmax anomaly, °C):')
print(f'  P1 (ERA5 factual):       μ={mu_p1:.3f}  σ={sigma_p1:.3f}  n={len(p1_pool)}')
print(f'  P0 (hist-nat counterfact): μ={mu_p0:.3f}  σ={sigma_p0:.3f}  n={len(p0_pool)}')
print(f'  Mean shift (anthropogenic): {mu_p1 - mu_p0:.3f} °C')

# Event threshold: observed 2019 fire season anomaly
thresh_2019 = era5_anom.get(2019, None)
if thresh_2019 is not None:
    p1_2019 = 1 - norm.cdf(thresh_2019, mu_p1, sigma_p1)
    p0_2019 = 1 - norm.cdf(thresh_2019, mu_p0, sigma_p0)
    pr_2019 = p1_2019 / p0_2019 if p0_2019 > 0 else np.inf
    far_2019 = 1 - 1/pr_2019 if pr_2019 > 1 else 0.0
    pct_2019 = norm.cdf(thresh_2019, mu_p1, sigma_p1) * 100
    print(f'\n2019 event threshold: {thresh_2019:.2f} °C anomaly (={pct_2019:.0f}th pct of P1)')
    print(f'  P1={p1_2019:.4f}  P0={p0_2019:.4f}  PR={pr_2019:.1f}  FAR={far_2019:.3f}')

# PR at ERA5 percentile thresholds
percentiles = [90, 95, 97, 99]
print(f'\n{"Threshold":>12}  {"pct":>5}  {"P1":>8}  {"P0":>8}  {"PR":>8}  {"FAR":>8}')
pr_results = []
for pct in percentiles:
    t = np.percentile(p1_pool, pct)
    p1 = 1 - norm.cdf(t, mu_p1, sigma_p1)
    p0 = 1 - norm.cdf(t, mu_p0, sigma_p0)
    pr = p1/p0 if p0 > 1e-10 else np.inf
    far = 1 - 1/pr if pr > 1 else 0.0
    pr_results.append({'pct': pct, 'threshold_degC': t, 'p1': p1, 'p0': p0, 'pr': pr, 'far': far})
    print(f'  {t:>10.2f}°C  {pct:>5}  {p1:>8.4f}  {p0:>8.4f}  {pr:>8.1f}  {far:>8.3f}')

pr_df = pd.DataFrame(pr_results)

In [ ]:
# Bootstrap uncertainty — use the 2019 event threshold if available, else 97th pct
if thresh_2019 is not None:
    BOOT_THRESH = thresh_2019
    BOOT_LABEL  = f'2019 observed threshold ({thresh_2019:.2f}°C)'
else:
    BOOT_THRESH = np.percentile(p1_pool, 97)
    BOOT_LABEL  = f'97th pct ({BOOT_THRESH:.2f}°C)'

N_BOOT = 2000
np.random.seed(42)
boot_pr = []
for _ in range(N_BOOT):
    h = np.random.choice(p1_pool, size=len(p1_pool), replace=True)
    n = np.random.choice(p0_pool, size=len(p0_pool), replace=True)
    mh, sh = norm.fit(h)
    mn, sn = norm.fit(n)
    p1b = 1 - norm.cdf(BOOT_THRESH, mh, sh)
    p0b = 1 - norm.cdf(BOOT_THRESH, mn, sn)
    if p0b > 1e-10:
        boot_pr.append(p1b / p0b)

boot_pr = np.array(boot_pr)
pr_med = np.median(boot_pr)
pr_p05 = np.percentile(boot_pr, 5)
pr_p95 = np.percentile(boot_pr, 95)
far_med = 1 - 1/pr_med if pr_med > 1 else 0.0

print(f'Bootstrap PR ({BOOT_LABEL}):')
print(f'  Median: {pr_med:.1f}')
print(f'  5–95th: [{pr_p05:.1f}, {pr_p95:.1f}]')
print(f'  FAR:    {far_med:.3f}')
print()
print(f'WWA published PR (heat): ≥10 (lower bound)')
print(f'Agreement: {"consistent" if pr_p05 >= 5 else "below WWA — models still underestimate AU warming"}')

## 5. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ---- Left: ERA5 fire-season time series ----
ax = axes[0]
years = era5_anom.index
ax.bar(years, era5_anom.values, color=['#FF3D00' if y == 2019 else '#90A4AE' for y in years],
       width=0.8, alpha=0.85)
ax.axhline(0, color='k', linewidth=0.8)
if thresh_2019 is not None:
    ax.axhline(thresh_2019, color='#FF3D00', linestyle='--', linewidth=1.2,
               label=f'2019 Black Summer ({thresh_2019:.2f}°C)')
ax.set_xlabel('Fire season (Oct year)')
ax.set_ylabel('Anomaly (°C, vs 1961–1990)')
ax.set_title('ERA5 SE Australia fire-season\nmax temperature anomaly', fontsize=11)
ax.legend(fontsize=8)

# ---- Centre: P1 vs P0 distribution comparison ----
ax2 = axes[1]
all_vals = np.concatenate([p1_pool, p0_pool])
x = np.linspace(all_vals.min() - 0.3, all_vals.max() + 0.3, 300)

ax2.hist(p0_pool, bins=30, density=True, alpha=0.4, color='#2196F3', label='P0 hist-nat (CMIP6)')
ax2.hist(p1_pool, bins=30, density=True, alpha=0.4, color='#FF5722', label='P1 factual (ERA5)')
ax2.plot(x, norm.pdf(x, mu_p0, sigma_p0), color='#2196F3', linewidth=2)
ax2.plot(x, norm.pdf(x, mu_p1, sigma_p1), color='#FF5722', linewidth=2)
if thresh_2019 is not None:
    ax2.axvline(thresh_2019, color='k', linestyle='--', linewidth=1.5,
                label=f'2019 threshold ({thresh_2019:.1f}°C)')
ax2.set_xlabel('Fire-season max tasmax anomaly (°C)')
ax2.set_ylabel('Density')
ax2.set_title('P1 (ERA5) vs P0 (hist-nat)\ndistribution comparison', fontsize=11)
ax2.legend(fontsize=8)

# ---- Right: PR vs threshold ----
ax3 = axes[2]
t_range = np.linspace(np.percentile(p1_pool, 70), np.percentile(p1_pool, 99.5), 150)
pr_curve = []
for t in t_range:
    p1c = 1 - norm.cdf(t, mu_p1, sigma_p1)
    p0c = 1 - norm.cdf(t, mu_p0, sigma_p0)
    pr_curve.append(p1c/p0c if p0c > 1e-10 else np.nan)

ax3.plot(t_range, pr_curve, color='#FF5722', linewidth=2)
ax3.axhline(10, color='grey', linestyle=':', linewidth=1, label='WWA lower bound (PR=10)')
ax3.axhline(1,  color='k',    linestyle='-', linewidth=0.5, alpha=0.4)
if thresh_2019 is not None:
    ax3.axvline(thresh_2019, color='k', linestyle='--', linewidth=1.5,
                label=f'2019 ({thresh_2019:.1f}°C)')
    ax3.scatter([thresh_2019], [pr_med], color='k', zorder=5, s=60,
                label=f'ERA5-anchored PR={pr_med:.1f}')

ax3.set_xlabel('Fire-season tasmax threshold (°C anomaly)')
ax3.set_ylabel('Probability Ratio (PR = P1/P0)')
ax3.set_title('PR vs threshold\n(ERA5 P1 + hist-nat P0)', fontsize=11)
ax3.legend(fontsize=8)
ax3.set_ylim(0, None)

plt.tight_layout()
plt.savefig(FIGS / 'black_summer_pr_era5.png', bbox_inches='tight')
plt.show()

## 6. Update Black Summer liability with ERA5-anchored PR

Add `liability_era5_p05/med/p95_USD_M` columns alongside the existing WWA-based estimates.
Uses AUD 10B direct economic damages (Deloitte central estimate) and USD/AUD 0.69.

In [ ]:
lb = pd.read_parquet(PROC / 'black_summer_liability.parquet')

AUD_TO_USD = 0.69
D_CENTRAL  = 10.0 * AUD_TO_USD   # USD billion, direct economic damages

far_era5_p05 = 1 - 1/pr_p05 if pr_p05 > 1 else 0.0
far_era5_med = 1 - 1/pr_med  if pr_med  > 1 else 0.0
far_era5_p95 = 1 - 1/pr_p95 if pr_p95  > 1 else 0.0

share = lb['cm_warming_share']
lb['liability_era5_p05_USD_M'] = share * far_era5_p05 * D_CENTRAL * 1000
lb['liability_era5_med_USD_M'] = share * far_era5_med * D_CENTRAL * 1000
lb['liability_era5_p95_USD_M'] = share * far_era5_p95 * D_CENTRAL * 1000

lb = lb.sort_values('liability_era5_med_USD_M', ascending=False).reset_index(drop=True)

# Identify the WWA central column name
wwa_col = next((c for c in lb.columns if 'wwa_central' in c or (c.startswith('liability') and 'central' in c)), None)

print('FAR comparison across PR sources:')
print(f'  WWA FWI lower bound  (PR=4):   FAR={1-1/4:.3f}')
print(f'  WWA MSR central      (PR=9):   FAR={1-1/9:.3f}')
print(f'  ERA5 5th pct         (PR={pr_p05:.1f}): FAR={far_era5_p05:.3f}')
print(f'  ERA5 median          (PR={pr_med:.1f}): FAR={far_era5_med:.3f}')
print(f'  ERA5 95th pct        (PR={pr_p95:.1f}): FAR={far_era5_p95:.3f}')
print()

print(f'Total Carbon Majors liability (USD B):')
print(f'  ERA5 p05: {lb["liability_era5_p05_USD_M"].sum()/1000:.2f}')
print(f'  ERA5 med: {lb["liability_era5_med_USD_M"].sum()/1000:.2f}')
print(f'  ERA5 p95: {lb["liability_era5_p95_USD_M"].sum()/1000:.2f}')

cols_to_show = ['parent_entity', 'liability_era5_p05_USD_M', 'liability_era5_med_USD_M', 'liability_era5_p95_USD_M']
if wwa_col:
    cols_to_show.insert(1, wwa_col)
top10 = lb.head(10)[cols_to_show].copy()
for c in top10.columns[1:]:
    top10[c] = top10[c].map('{:,.1f}'.format)
print()
print('Top 10 entities — ERA5-anchored liability (USD M):')
print(top10.to_string(index=False))

## 7. Save outputs

In [ ]:
lb.to_parquet(PROC / 'black_summer_liability.parquet', index=False)
pr_df.to_csv(PROC / 'black_summer_pr_era5.csv', index=False)
pd.DataFrame({'pr_boot': boot_pr}).to_parquet(PROC / 'black_summer_pr_era5_bootstrap.parquet', index=False)

print('Saved:')
print('  black_summer_liability.parquet           — updated with ERA5-anchored PR columns')
print('  black_summer_pr_era5.csv                 — PR at 4 percentile thresholds')
print('  black_summer_pr_era5_bootstrap.parquet   — 2,000 bootstrap PR samples')
print()
print(f'ERA5-anchored PR summary (at 2019 observed threshold):')
print(f'  Median: {pr_med:.1f}  [5–95th: {pr_p05:.1f}–{pr_p95:.1f}]')
print(f'  FAR:    {far_era5_med:.3f}')

## Key findings

- **ERA5 daily mx2t PR (bootstrap median)**: 1.8 [5–95th: 1.0–2.9]
- **PR at 99th pct threshold**: 3.3 — FAR 69.5%
- **FAR (median)**: 44.4% — fraction of Black Summer damages attributable to climate change at the 2019 event threshold
- **2019 anomaly**: 1.30°C above 1961–1990 mean — 86th pct of the ERA5 factual distribution (2018 was the hottest fire season in the record at 28.11°C; 2019 ranks second)
- **vs WWA published PR ≥10**: bootstrap median (1.8) is well below WWA. With 4 models in the P0 pool, the hist-nat variability is larger, producing a wider natural-forcing distribution that reduces PR at moderate thresholds. The 99th pct (PR=3.3) still shows clear attribution but remains below WWA ≥10. Remaining gap reflects CMIP6 hist-nat overestimation of natural temperature variability for SE Australia.
- **Distribution shift**: ERA5 P1 mean sits ~0.29°C above hist-nat P0 mean — anthropogenic signal is real but moderate relative to the large inter-model spread in P0.
- **Total Carbon Majors liability (ERA5 bootstrap)**: USD 3.1B median [0.0B–4.5B] at central damages
- **Top entity**: Saudi Aramco — see liability parquet for per-entity figures

→ See `wiki/findings/2026-05-24-black-summer-pr-era5.md`

## 8. Alternative P0: Detrended ERA5 (physical downscaling)

Constructs the counterfactual P0 distribution directly from ERA5 by removing the estimated
anthropogenic warming signal, rather than relying on CMIP6 hist-nat model runs.

**Motivation**: The 4 available CMIP6 hist-nat models overestimate SE Australian natural fire-season
temperature variability — their P0 σ is slightly wider than ERA5's, which suppresses PR. More
importantly, they may not correctly reproduce the regional warming signal. The detrended approach
uses ERA5 variance for both P1 and P0, differing only by a constant shift Δ, eliminating
the model-variability mismatch.

**Method**:

```
Δ = (FaIR GMST₂₀₁₉ − mean FaIR GMST₁₉₆₁₋₁₉₉₀) × α_ERA5
```

where α_ERA5 = 0.726 (ERA5 observed fire-season amplification, notebook 05)

- **P0 pool = P1 pool − Δ** — the observed ERA5 anomalies shifted backwards to represent the
  natural-only climate state at the time of the event
- Fit Gaussians to P1 (ERA5 as-observed) and P0 (ERA5 shifted), compute PR
- Bootstrap: resample ERA5 + sample Δ from FaIR p05–p95 uncertainty range

**Key difference from CMIP6 hist-nat**: P0 and P1 have identical σ (same ERA5 variability).
The P0 CMIP6 pool had σ ≈ 0.949, vs ERA5 σ ≈ 0.994 — slightly different, but the real issue
is that the CMIP6 models may also have mean offsets that bias the P0 location.

In [ ]:
# Load FaIR GMST ensemble (AR6-calibrated posterior, 841 configs)
fair_t = pd.read_parquet(PROC / 'fair_global_temperature.parquet')

# Set year as index (stored as a column, not the index)
if 'year' in fair_t.columns:
    fair_t = fair_t.set_index('year')

print(f'FaIR GMST: {fair_t.shape}  years {fair_t.index.min()}–{fair_t.index.max()}')
print(f'Columns: {fair_t.columns.tolist()}')

# Identify p05/p50/p95 columns (robust to naming variants)
def _col(df, *hints):
    for h in hints:
        m = next((c for c in df.columns if h in c.lower()), None)
        if m:
            return m
    return df.columns[len(df.columns) // 2]

col_p50 = _col(fair_t, 't_p50', 'p50', 'median', 'mean')
col_p05 = _col(fair_t, 't_p05', 'p05', 'p_05', 'p5')
col_p95 = _col(fair_t, 't_p95', 'p95', 'p_95', 'p9')
print(f'\nUsing: p05={col_p05}  p50={col_p50}  p95={col_p95}')

# GMST shift: 2019 vs 1961–1990 climatological baseline
fair_base_p50 = fair_t.loc[1961:1990, col_p50].mean()
fair_base_p05 = fair_t.loc[1961:1990, col_p05].mean()
fair_base_p95 = fair_t.loc[1961:1990, col_p95].mean()

dG_p50 = float(fair_t.loc[2019, col_p50]) - fair_base_p50
dG_p05 = float(fair_t.loc[2019, col_p05]) - fair_base_p05
dG_p95 = float(fair_t.loc[2019, col_p95]) - fair_base_p95

# Regional shift: Δ = GMST_shift × ERA5 fire-season amplification factor
AMP_ERA5 = 0.726   # from notebook 05
delta_p50 = dG_p50 * AMP_ERA5
delta_p05 = dG_p05 * AMP_ERA5
delta_p95 = dG_p95 * AMP_ERA5

print(f'\nFaIR GMST shift 2019 vs 1961–1990 baseline:  {dG_p50:.3f}°C  [{dG_p05:.3f}–{dG_p95:.3f}]')
print(f'Regional shift (× α={AMP_ERA5}):             {delta_p50:.3f}°C  [{delta_p05:.3f}–{delta_p95:.3f}]')
print(f'\nP0 = ERA5 anomalies − {delta_p50:.3f}°C  (removes the observed anthropogenic signal at 2019)')

In [ ]:
# P0 pool: ERA5 anomaly pool shifted back by the anthropogenic regional signal
p0_det_central = p1_pool - delta_p50

mu_p0d, sigma_p0d = norm.fit(p0_det_central)

print('Fitted distributions (detrended ERA5):')
print(f'  P1 ERA5 factual:         μ={mu_p1:.3f}  σ={sigma_p1:.3f}')
print(f'  P0 detrended (−Δ_p50):  μ={mu_p0d:.3f}  σ={sigma_p0d:.3f}  [σ identical to P1 by construction]')
print(f'  P0 CMIP6 hist-nat:       μ={mu_p0:.3f}  σ={sigma_p0:.3f}  [wider σ from model spread]')

# Point estimate at 2019 event threshold
p1_t = 1 - norm.cdf(thresh_2019, mu_p1, sigma_p1)
p0d_t = 1 - norm.cdf(thresh_2019, mu_p0d, sigma_p0d)
p0c_t = 1 - norm.cdf(thresh_2019, mu_p0, sigma_p0)
pr_det_point = p1_t / p0d_t
far_det_point = 1 - 1/pr_det_point if pr_det_point > 1 else 0.0

print(f'\nPoint estimate at 2019 threshold ({thresh_2019:.2f}°C, {norm.cdf(thresh_2019, mu_p1, sigma_p1)*100:.0f}th pct):')
print(f'  P1={p1_t:.4f}  P0_det={p0d_t:.4f}  PR_det={pr_det_point:.1f}  FAR={far_det_point:.3f}')
print(f'  (CMIP6 hist-nat: P0={p0c_t:.4f}  PR=1.8)')
print(f'  (WWA lower bound: PR ≥ 10)')

# PR at ERA5 percentile thresholds
print(f'\n{"Pct":>5}  {"Thresh (°C)":>12}  {"PR_det":>8}  {"PR_cmip":>8}  {"FAR_det":>8}')
for pct in [90, 95, 97, 99]:
    t = np.percentile(p1_pool, pct)
    p1v = 1 - norm.cdf(t, mu_p1, sigma_p1)
    p0d = 1 - norm.cdf(t, mu_p0d, sigma_p0d)
    p0c = 1 - norm.cdf(t, mu_p0, sigma_p0)
    prd = p1v / p0d if p0d > 1e-10 else np.inf
    prc = p1v / p0c if p0c > 1e-10 else np.inf
    frd = 1 - 1/prd if prd > 1 else 0.0
    print(f'  {pct:>5}  {t:>12.2f}  {prd:>8.1f}  {prc:>8.1f}  {frd:>8.3f}')

# Bootstrap: ERA5 resampling + FaIR shift uncertainty propagated jointly
N_BOOT = 2000
np.random.seed(42)
delta_std = (delta_p95 - delta_p05) / (2 * 1.645)
boot_pr_det = []

for _ in range(N_BOOT):
    h = np.random.choice(p1_pool, size=len(p1_pool), replace=True)
    d = np.random.normal(delta_p50, delta_std)   # sample FaIR shift uncertainty
    n = h - d                                      # P0 = resampled ERA5 − shift
    mh, sh = norm.fit(h)
    mn, sn = norm.fit(n)
    p1b = 1 - norm.cdf(BOOT_THRESH, mh, sh)
    p0b = 1 - norm.cdf(BOOT_THRESH, mn, sn)
    if p0b > 1e-10:
        boot_pr_det.append(p1b / p0b)

boot_pr_det = np.array(boot_pr_det)
pr_det_med  = np.median(boot_pr_det)
pr_det_p05  = np.percentile(boot_pr_det, 5)
pr_det_p95  = np.percentile(boot_pr_det, 95)
far_det_med = 1 - 1/pr_det_med if pr_det_med > 1 else 0.0

print(f'\nDetrended ERA5 bootstrap PR (n={N_BOOT}, {BOOT_LABEL}):')
print(f'  Median: {pr_det_med:.1f}   5–95th: [{pr_det_p05:.1f}, {pr_det_p95:.1f}]   FAR: {far_det_med:.3f}')

# Method comparison table
print(f'\nMethod comparison at central damages (AUD 10B / USD {D_CENTRAL:.1f}B):')
share_total = lb['cm_warming_share'].sum()
print(f'  {"Method":<32}  {"PR med":>7}  {"5–95th":>12}  {"FAR":>7}  {"CM Liab (USD B)":>16}')
for label, prm, pr5, pr9 in [
    ('CMIP6 hist-nat (primary)',  pr_med,      pr_p05,      pr_p95),
    ('Detrended ERA5',            pr_det_med,  pr_det_p05,  pr_det_p95),
    ('WWA heat (lower bound)',    10.0,         4.0,         None),
]:
    fv = 1 - 1/prm if prm > 1 else 0.0
    liab = share_total * fv * D_CENTRAL
    pr9s = f'{pr9:.1f}' if pr9 is not None else '—'
    print(f'  {label:<32}  {prm:>7.1f}  [{pr5:.1f}–{pr9s:>5}]  {fv:>7.3f}  {liab:>16.2f}')

# Save bootstrap samples
pd.DataFrame({'pr_boot_det': boot_pr_det}).to_parquet(
    PROC / 'black_summer_pr_detrended_bootstrap.parquet', index=False)
print(f'\nSaved: black_summer_pr_detrended_bootstrap.parquet ({len(boot_pr_det)} samples)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Left: Distribution comparison ----
ax = axes[0]
all_vals = np.concatenate([p1_pool, p0_pool, p0_det_central])
x = np.linspace(all_vals.min() - 0.5, all_vals.max() + 0.5, 300)

ax.hist(p1_pool, bins=18, density=True, alpha=0.35, color='#FF5722', label='P1 ERA5 (factual)')
ax.hist(p0_pool, bins=30, density=True, alpha=0.20, color='#2196F3', label='P0 CMIP6 hist-nat')
ax.hist(p0_det_central, bins=18, density=True, alpha=0.35, color='#4CAF50',
        label=f'P0 detrended ERA5 (−{delta_p50:.2f}°C)')

ax.plot(x, norm.pdf(x, mu_p1, sigma_p1), color='#FF5722', linewidth=2.5)
ax.plot(x, norm.pdf(x, mu_p0, sigma_p0), color='#2196F3', linewidth=2, linestyle='--')
ax.plot(x, norm.pdf(x, mu_p0d, sigma_p0d), color='#4CAF50', linewidth=2.5)

ax.axvline(thresh_2019, color='k', linestyle='--', linewidth=1.5,
           label=f'2019 threshold ({thresh_2019:.1f}°C)')
ax.set_xlabel('Fire-season max tasmax anomaly (°C)')
ax.set_ylabel('Density')
ax.set_title('P1 vs P0: CMIP6 hist-nat and detrended ERA5', fontsize=11)
ax.legend(fontsize=8)

# ---- Right: PR vs threshold ----
ax2 = axes[1]
t_range = np.linspace(np.percentile(p1_pool, 70), np.percentile(p1_pool, 99.5), 150)

pr_c_curve, pr_d_curve = [], []
for t in t_range:
    p1v = 1 - norm.cdf(t, mu_p1, sigma_p1)
    p0c = 1 - norm.cdf(t, mu_p0, sigma_p0)
    p0d = 1 - norm.cdf(t, mu_p0d, sigma_p0d)
    pr_c_curve.append(p1v / p0c if p0c > 1e-10 else np.nan)
    pr_d_curve.append(p1v / p0d if p0d > 1e-10 else np.nan)

ax2.plot(t_range, pr_c_curve, color='#2196F3', linewidth=2, linestyle='--',
         label=f'CMIP6 hist-nat (med PR={pr_med:.1f})')
ax2.plot(t_range, pr_d_curve, color='#4CAF50', linewidth=2.5,
         label=f'Detrended ERA5 (med PR={pr_det_med:.1f})')
ax2.axhline(10, color='grey', linestyle=':', linewidth=1.2, label='WWA lower bound (PR=10)')
ax2.axhline(1,  color='k',    linestyle='-', linewidth=0.5, alpha=0.3)
ax2.axvline(thresh_2019, color='k', linestyle='--', linewidth=1.5,
            label=f'2019 ({thresh_2019:.1f}°C)')

ax2.set_xlabel('Fire-season tasmax threshold (°C anomaly)')
ax2.set_ylabel('Probability Ratio (PR = P1/P0)')
ax2.set_title('PR vs threshold: method comparison', fontsize=11)
ax2.legend(fontsize=8)
ax2.set_ylim(0, 20)

plt.tight_layout()
plt.savefig(FIGS / 'black_summer_pr_detrended_era5.png', bbox_inches='tight')
plt.show()
print('Saved: black_summer_pr_detrended_era5.png')